## Fisher's iris flower data

**Context**

This classic dataset contains measurements of iris flowers, along with what type of flower they are. The idea is to classify whether the flower is a Versicolor (0), Setosa (1), or Virginica (2).  

<img src="iris.png" alt="Iris flower">  

# Reading the dataset

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

# Convert it into a Pandas dataframe
iris_data = pd.DataFrame(data= np.c_[iris['data'], iris['target']],
                     columns= iris['feature_names'] + ['target'])

# Understanding the dataset

I'm assuming that you've done some basic exploration, plotted the data, etc. Here are the essentials.

In [ ]:
iris_data.head(100)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0.0
1,4.9,3.0,1.4,0.2,0.0
2,4.7,3.2,1.3,0.2,0.0
3,4.6,3.1,1.5,0.2,0.0
4,5.0,3.6,1.4,0.2,0.0
...,...,...,...,...,...
95,5.7,3.0,4.2,1.2,1.0
96,5.7,2.9,4.2,1.3,1.0
97,6.2,2.9,4.3,1.3,1.0
98,5.1,2.5,3.0,1.1,1.0


In [ ]:
iris_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    float64
dtypes: float64(5)
memory usage: 6.0 KB


In [ ]:
iris_data.target = iris_data.target.astype(int)

In [ ]:
iris_data.describe(include='all')

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


In [ ]:
print('Covariance:')
iris_data.cov()

Covariance:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
sepal length (cm),0.685694,-0.042434,1.274315,0.516271,0.530872
sepal width (cm),-0.042434,0.189979,-0.329656,-0.121639,-0.152349
petal length (cm),1.274315,-0.329656,3.116278,1.295609,1.372483
petal width (cm),0.516271,-0.121639,1.295609,0.581006,0.597315
target,0.530872,-0.152349,1.372483,0.597315,0.671141


In [ ]:
iris_data.target.value_counts()

0    50
1    50
2    50
Name: target, dtype: int64

Looks like the dataset is already well balanced. Let's check for one type of missing data.

In [ ]:
print( iris_data.isnull().sum() )

sepal length (cm)    0
sepal width (cm)     0
petal length (cm)    0
petal width (cm)     0
target               0
dtype: int64


We will keep things simple here and do a binary classification. So we will classify as either Versicolor, or Setosa. This means that we can use a Logistic Regression for our representation. If you wanted to classify all three, you would need to use Softmax Regression.

In [ ]:
iris_data = iris_data[ iris_data['target'] < 2 ]
iris_data.target.value_counts()

0    50
1    50
Name: target, dtype: int64

We now need to split into train and test sets. For this, we use a stratified split. This tries to keep the same ratio of each target category in the train and test sets. We're doing this because the dataset is small.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

ss = StratifiedShuffleSplit(n_splits = 1, test_size=0.3)
ss.get_n_splits(iris_data)

for train_index, test_index in ss.split(iris_data, iris_data['target']):
    print('Train: ', train_index)
    train_data = iris_data.loc[train_index]
    print('Test:', test_index)
    test_data = iris_data.loc[test_index]

Train:  [30 80  5 82 28 54 49 67 57 62 92 29 43  6 63  4 39 56 16 96 50 68 25 13
 70 79 59 26 95 11  7 71 12  1  0 85 61 66 47 53 46 91 48 15 64  8 23 76
 52 20 32 98 69  2 72 45 89 44 34 58 21  9 51 87 75  3 84 27 42 93]
Test: [81 17 97 83 22 55 99 31 40 37 18 60 24 33 38 41 90 73 78 36 74 14 77 35
 10 88 94 65 19 86]


In [ ]:
train_data.target.value_counts()

0    35
1    35
Name: target, dtype: int64

In [ ]:
test_data.target.value_counts()

1    15
0    15
Name: target, dtype: int64

Divide into target and features

In [ ]:
y_train = iris_data['target']
X_train = iris_data.drop(labels='target', axis=1)
y_test = iris_data['target']
X_test = iris_data.drop(labels='target', axis=1)

# Evolutionary algorithm

In [ ]:
!pip install deap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 2.7 MB/s eta 0:00:00


In [ ]:
from deap import base
from deap import creator
from deap import tools
import random

In [ ]:
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

In [ ]:
toolbox = base.Toolbox()
toolbox.register("attr_float", random.uniform, -1.0, 1.0)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, 5)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

We need to decide on a representation for our individuals, and will use the GA to optimise a regression model. This means that an individual simply represents the weights with which to multiply the features.

For continuous features you can capture non-linearities. Which you can do e.g. using polynomials. Here, we will just keep it simple to illustrate the concept.

This means that we will need individuals to represent 1 weight for each of the four features in the dataset. It is also a good idea to add an intercept. Thus, individuals will have five genes.

We need to define a function that gives a classification for an individual, on a given row in the dataset. For that we use a logistic regression. However, because we want a first prediction (0 or 1) rather than probability, we apply a threshold at 0, to make it a binary regression.

In [ ]:
import math

def sigmoid(x):
  return 1 / (1 + math.exp(-x))

def prediction(individual,row):
    regr = individual[0] + sum(i * r for i , r in zip(individual[1:4], row ) )
    return( sigmoid(regr) )

This represents the model:  

y = a + x1w1 + x2w2 + x3w3 + x4w4

y = (sigmoid(y))

Where a is the intercept xi is the feature/column and wi is the weight from the individual that we are evolving.  

In [ ]:
def evalAccuracyTrain(individual, X_train, y_train):
    y_pred = [None] * X_train.shape[0]

    index = 0
    for i, row in X_train.iterrows():
        y_pred[index] = prediction(individual,row)
        index += 1

    # calculate mean squared difference (using the training set)
    diff = (np.array(y_pred) - np.array(y_train)) ** 2

    return( sum(diff), )

Normally logistic regression does not use mean squared error (commonly used in linear regression) because it makes the landscape have some local optima. However, this will be fine for this problem.

In [ ]:
toolbox.register("evaluate", evalAccuracyTrain)

Let's test our evaluation function before we use it

In [ ]:
import random
indv1 = [random.random() for iter in range(5)]

In [ ]:
y_pred = [None] * iris_data.shape[0]

In [ ]:
prediction(indv1,X_train.iloc[50])

0.99985973687342

In [ ]:
prediction(indv1,X_train.iloc[50]) > 0.5

True

Let's carry on with the GA

Now we register our operators. We are using a per-gene probability of mutation here, and we have 6 genes. So ba careful not to have mutation so high that you can almost always mutate an individual. You need to preserve some of what you have found.

In [ ]:
toolbox.register("mutate", tools.mutGaussian, mu=0.0, sigma=0.4, indpb=0.2)
toolbox.register("mate", tools.cxOnePoint)
toolbox.register("select", tools.selTournament, tournsize=3)

In [ ]:
NGEN, CXPB = 30, 0.2
pop = toolbox.population(n=200)

fitnesses = [toolbox.evaluate(ind, X_train, y_train) for ind in pop]
for ind, fit in zip(pop, fitnesses):
    ind.fitness.values = fit

In [ ]:
for g in range(NGEN):
    print("-- Generation %i --" % g)
    offspring = toolbox.select(pop, len(pop))
    offspring = list(map(toolbox.clone, offspring))

    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < CXPB:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    for mutant in offspring:
        toolbox.mutate(mutant)
        del mutant.fitness.values

    invalid_inds = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = [toolbox.evaluate(ind, X_train, y_train) for ind in invalid_inds]
    for ind, fit in zip(invalid_inds, fitnesses):
        ind.fitness.values = fit

    best_ind = tools.selBest(pop, 1)[0]
    print("Best individual is %s, %s" % (best_ind, best_ind.fitness.values))
    pop[:] = offspring

-- Generation 0 --
Best individual is [-0.1971678606963223, 0.03588256017785807, -0.7407886917081481, 0.6569106237901579, 0.16669414619202527], (6.8601330651362185,)
-- Generation 1 --
Best individual is [-0.7478994632890557, 0.447840340359682, -1.4184658728419386, 0.7680348849364738, -0.11278485088374346], (3.589360333525837,)
-- Generation 2 --
Best individual is [-0.7478994632890557, 0.447840340359682, -1.4184658728419386, 0.8726521816789242, -0.11278485088374346], (2.3535435704359355,)
-- Generation 3 --
Best individual is [-0.7478994632890557, 0.447840340359682, -1.4184658728419386, 0.952875874367656, -0.11278485088374346], (1.8888361936399547,)
-- Generation 4 --
Best individual is [-0.6995795883769962, -0.8028688813618163, -0.22783092150565398, 2.038783879951472, -0.1831971919021198], (0.8593743451589344,)
-- Generation 5 --
Best individual is [-0.6995795883769962, -0.8028688813618163, -0.22783092150565398, 2.038783879951472, -0.1831971919021198], (0.8593743451589344,)
-- Genera

# Evaluate

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = [None] * X_test.shape[0]
index = 0
for i, row in X_test.iterrows():
    y_pred[index] = prediction(best_ind,row)
    index += 1

y_bin_pred = np.array(y_pred) > 0.5
accuracy = accuracy_score(y_test, y_bin_pred)
print("Accuracy: %.2f%%" % (accuracy * 100.0))


Accuracy: 100.00%
